# S3.1 — Paper ORM Model & Repository

Interactive verification of the Paper model, repository CRUD, and Pydantic schemas using an in-memory SQLite database.

In [1]:
import sys
sys.path.insert(0, "../..")

from datetime import UTC, datetime
from sqlalchemy import inspect
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker, create_async_engine

from src.db.base import Base
from src.models.paper import Paper
from src.schemas.paper import PaperCreate, PaperUpdate, PaperResponse
from src.repositories.paper import PaperRepository

print("✓ All imports successful")

✓ All imports successful


## 1. Paper Model Inspection

Verify the Paper ORM model has the correct table name, columns, and inherits from Base.

In [2]:
# Verify table name
assert Paper.__tablename__ == "papers"
print(f"Table name: {Paper.__tablename__}")

# Verify inheritance
assert issubclass(Paper, Base)
print(f"Inherits from Base: True")

# Verify all columns
mapper = inspect(Paper)
column_names = sorted(c.key for c in mapper.column_attrs)
print(f"\nColumns ({len(column_names)}):")
for col in column_names:
    print(f"  - {col}")

expected = {
    "id", "arxiv_id", "title", "authors", "abstract", "categories",
    "published_date", "updated_date", "pdf_url", "pdf_content",
    "sections", "parsing_status", "parsing_error", "created_at", "updated_at",
}
assert expected.issubset(set(column_names)), f"Missing: {expected - set(column_names)}"
print("\n✓ Paper model structure verified")

Table name: papers
Inherits from Base: True

Columns (15):
  - abstract
  - arxiv_id
  - authors
  - categories
  - created_at
  - id
  - parsing_error
  - parsing_status
  - pdf_content
  - pdf_url
  - published_date
  - sections
  - title
  - updated_at
  - updated_date

✓ Paper model structure verified


## 2. Pydantic Schemas

Verify PaperCreate, PaperUpdate, and PaperResponse schemas.

In [3]:
# PaperCreate — required fields with defaults
data = PaperCreate(
    arxiv_id="2401.00001",
    title="Attention Is All You Need",
    authors=["Vaswani", "Shazeer", "Parmar"],
    abstract="The dominant sequence transduction models are based on complex recurrent or convolutional neural networks.",
    categories=["cs.CL", "cs.AI"],
    published_date=datetime(2017, 6, 12, tzinfo=UTC),
    pdf_url="https://arxiv.org/pdf/1706.03762v1",
)
assert data.parsing_status == "pending"
print(f"PaperCreate: arxiv_id={data.arxiv_id}, status={data.parsing_status}")

# PaperUpdate — all fields optional
update = PaperUpdate(title="Updated Title")
dump = update.model_dump(exclude_unset=True)
assert dump == {"title": "Updated Title"}
print(f"PaperUpdate partial dump: {dump}")

# PaperResponse — from_attributes enabled
assert PaperResponse.model_config.get("from_attributes") is True
print(f"PaperResponse from_attributes: True")

print("\n✓ Pydantic schemas verified")

PaperCreate: arxiv_id=2401.00001, status=pending
PaperUpdate partial dump: {'title': 'Updated Title'}
PaperResponse from_attributes: True

✓ Pydantic schemas verified


## 3. Repository CRUD (In-Memory SQLite)

Create an in-memory database and test all repository operations.

In [4]:
# Setup in-memory SQLite engine
engine = create_async_engine("sqlite+aiosqlite:///:memory:", echo=False)
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

session_factory = async_sessionmaker(bind=engine, class_=AsyncSession, expire_on_commit=False)
session = session_factory()

repo = PaperRepository(session)
print("✓ In-memory database and session created")

✓ In-memory database and session created


In [5]:
# CREATE — insert a paper
paper_data = PaperCreate(
    arxiv_id="1706.03762",
    title="Attention Is All You Need",
    authors=["Vaswani", "Shazeer", "Parmar", "Uszkoreit", "Jones", "Gomez", "Kaiser", "Polosukhin"],
    abstract="The dominant sequence transduction models are based on complex recurrent or convolutional neural networks.",
    categories=["cs.CL", "cs.AI"],
    published_date=datetime(2017, 6, 12, tzinfo=UTC),
    pdf_url="https://arxiv.org/pdf/1706.03762v1",
)
paper = await repo.create(paper_data)
await session.flush()

assert paper.id is not None
assert paper.arxiv_id == "1706.03762"
assert paper.parsing_status == "pending"
print(f"Created: {paper}")
print(f"  ID: {paper.id}")
print(f"  Authors: {paper.authors}")

print("\n✓ CREATE verified")

Created: <Paper arxiv_id='1706.03762' title='Attention Is All You Need...'>
  ID: 602cb7ac-aab4-4d9a-845a-827c35408561
  Authors: ['Vaswani', 'Shazeer', 'Parmar', 'Uszkoreit', 'Jones', 'Gomez', 'Kaiser', 'Polosukhin']

✓ CREATE verified


In [6]:
# READ — get by ID and arXiv ID
found_by_id = await repo.get_by_id(paper.id)
assert found_by_id is not None
assert found_by_id.title == "Attention Is All You Need"
print(f"get_by_id: {found_by_id.title}")

found_by_arxiv = await repo.get_by_arxiv_id("1706.03762")
assert found_by_arxiv is not None
print(f"get_by_arxiv_id: {found_by_arxiv.title}")

# exists
assert await repo.exists("1706.03762") is True
assert await repo.exists("nonexistent") is False
print(f"exists('1706.03762'): True")
print(f"exists('nonexistent'): False")

# not found
import uuid
assert await repo.get_by_id(uuid.uuid4()) is None
print(f"get_by_id(random UUID): None")

print("\n✓ READ verified")

get_by_id: Attention Is All You Need
get_by_arxiv_id: Attention Is All You Need
exists('1706.03762'): True
exists('nonexistent'): False
get_by_id(random UUID): None

✓ READ verified


In [7]:
# UPDATE — partial update
updated = await repo.update("1706.03762", PaperUpdate(title="Attention Is All You Need (Updated)"))
assert updated is not None
assert updated.title == "Attention Is All You Need (Updated)"
assert updated.authors == ["Vaswani", "Shazeer", "Parmar", "Uszkoreit", "Jones", "Gomez", "Kaiser", "Polosukhin"]
print(f"Updated title: {updated.title}")
print(f"Authors unchanged: {len(updated.authors)} authors")

# UPDATE parsing status
parsed = await repo.update_parsing_status(
    "1706.03762",
    status="success",
    pdf_content="Full paper text content here...",
    sections=[{"title": "Introduction", "content": "..."}],
)
assert parsed.parsing_status == "success"
assert parsed.pdf_content is not None
print(f"Parsing status: {parsed.parsing_status}")
print(f"Sections: {len(parsed.sections)} section(s)")

print("\n✓ UPDATE verified")

Updated title: Attention Is All You Need (Updated)
Authors unchanged: 8 authors
Parsing status: success
Sections: 1 section(s)

✓ UPDATE verified
